# Train MiniGPT on Kaggle (TinyStories + BPE)

Runs the full MiniGPT training pipeline on a **Kaggle GPU** — unattended via
**Save & Run All (Commit)**.

**Before you start (notebook settings, top right):**
- **Accelerator** → GPU T4 x2
- **Internet** → **On** (required for TinyStories download)
- **Persistence** → Files only (default)

**To train without keeping the tab open:**
1. Run the **Setup** cell once to verify GPU.
2. Click **Save Version** → **Save & Run All (Commit)**.
3. Close the browser — Kaggle runs the notebook in the cloud.
4. When the commit finishes, open **Output** and download `minigpt-artifacts/`.

> All modeling code lives in `src/` on GitHub — this notebook only calls `train()`.

## 1. Setup

In [ ]:
import os
import shutil
from pathlib import Path

import torch

REPO_URL = "https://github.com/Ilyes-Jamoussi/minigpt-llm.git"
WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "minigpt-llm"
ARTIFACTS_OUT = WORKING / "minigpt-artifacts"

if not REPO_DIR.joinpath("src").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    !git clone -q $REPO_URL $REPO_DIR

os.chdir(REPO_DIR)
!pip install -q safetensors datasets tqdm

print("working directory:", Path.cwd())
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU — enable GPU T4 in notebook settings before committing.")

## 2. Configure

Targets ~14M parameters and ~246M tokens (~5–9 h on one T4).
Lower `max_steps` if a commit hits the session time limit.

In [ ]:
import logging

from src.config import GPTConfig, TrainConfig, resolve_device

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
print("device:", resolve_device())

train_config = TrainConfig(
    dataset="tinystories",
    tokenizer_type="bpe",
    bpe_vocab_size=8192,
    max_chars=150_000_000,
    batch_size=32,
    max_steps=30_000,
    learning_rate=3e-4,
    min_lr=3e-5,
    warmup_steps=300,
    weight_decay=0.1,
    grad_clip=1.0,
    eval_interval=500,
    eval_iters=100,
    sample_interval=2_000,
    sample_prompt="Once upon a time",
    sample_max_new_tokens=200,
    seed=1337,
)

model_config = GPTConfig(
    vocab_size=train_config.bpe_vocab_size,
    block_size=256,
    n_layer=6,
    n_head=6,
    d_model=384,
    dropout=0.1,
)

## 3. Train

This cell takes **~5–9 hours**. When using **Save & Run All**, you do not need to watch it.

In [ ]:
from src.train import train

metrics = train(train_config, model_config)
metrics

## 4. Save artifacts to Kaggle Output

Copies the four model files to `/kaggle/working/minigpt-artifacts/` so they appear in the
notebook **Output** tab after a successful commit.

In [ ]:
import json
import shutil

from src.config import METRICS_FILENAME, MODELS_DIR

metrics_path = MODELS_DIR / METRICS_FILENAME
metrics_doc = json.loads(metrics_path.read_text())

required = {
    "dataset": "tinystories",
    "tokenizer_type": "bpe",
}
for key, expected in required.items():
    actual = metrics_doc.get(key)
    if actual != expected:
        raise ValueError(f"metrics {key}={actual!r}, expected {expected!r}")

weights = MODELS_DIR / "model.safetensors"
size_mb = weights.stat().st_size / 1e6
if size_mb < 40:
    raise ValueError(f"model.safetensors is only {size_mb:.1f} MB — expected ~55 MB for BPE model")

if ARTIFACTS_OUT.exists():
    shutil.rmtree(ARTIFACTS_OUT)
shutil.copytree(MODELS_DIR, ARTIFACTS_OUT)

print("Saved to:", ARTIFACTS_OUT)
print(json.dumps(metrics_doc, indent=2))
print("\nDownload minigpt-artifacts/ from the notebook Output tab when the commit finishes.")

## 5. Sample generations (optional)

In [ ]:
import torch

from src.config import MODELS_DIR, resolve_device
from src.model import load_pretrained

device = resolve_device()
model, tokenizer = load_pretrained(MODELS_DIR, device)

for prompt in ["Once upon a time", "The little robot", "Lily and Tom went to"]:
    ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(ids, max_new_tokens=200, temperature=0.8, top_k=50, top_p=0.95)
    print(tokenizer.decode(out[0].tolist()))
    print("-" * 60)